# MIT Humanoid — Pinocchio vs MuJoCo 사양 검증

`models/humanoid/`의 MIT humanoid URDF/MJCF가 동일한 로봇을 표현하는지 수치 비교.

- URDF: `urdf/humanoid_fixed_arms_sf_update.urdf` (10-DoF: 5 left + 5 right legs, arms fixed)
- MJCF: `xml/mit_humanoid_scene.xml`

## 변환 규약
| | MuJoCo | Pinocchio (FreeFlyer) |
|---|---|---|
| `qpos[3:7]` quaternion | `(w,x,y,z)` | `(x,y,z,w)` |
| free joint `qvel[0:3]` (linear) | **world frame** | **body local** |
| free joint `qvel[3:6]` (angular) | body local | body local |

Jacobian/M 비교 시 좌표 변환 행렬 `T = blkdiag(R_body_to_world, I_3, I_{nv-6})` 필수.

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[1]
sys.path.insert(0, str(_HERE))

import numpy as np
import mujoco
import mujoco.viewer
import pinocchio as pin
from pinocchio.robot_wrapper import RobotWrapper

%load_ext autoreload
%autoreload 2

URDF = str(_ROOT / 'models' / 'humanoid' / 'urdf' / 'humanoid_fixed_arms_sf_update.urdf')
PKG  = str(_ROOT / 'models' / 'humanoid')
MJCF = str(_ROOT / 'models' / 'humanoid' / 'xml' / 'mit_humanoid_scene.xml')

print(f'URDF: {URDF}')
print(f'MJCF: {MJCF}')

## 1. 두 모델 로딩 + 기본 정보 비교

In [ ]:
# Pinocchio (FreeFlyer)
robot = RobotWrapper.BuildFromURDF(URDF, package_dirs=[PKG],
                                    root_joint=pin.JointModelFreeFlyer())
pin_model = robot.model
pin_data  = robot.data

# MuJoCo
mj_model = mujoco.MjModel.from_xml_path(MJCF)
mj_data  = mujoco.MjData(mj_model)

m_pin = pin.computeTotalMass(pin_model)
m_mj  = float(mj_model.body_mass[1:].sum())  # world body(0) 제외

print(f'Pinocchio: nq={pin_model.nq}, nv={pin_model.nv}, njoints={pin_model.njoints}, mass={m_pin:.4f} kg')
print(f'MuJoCo   : nq={mj_model.nq}, nv={mj_model.nv}, nbody={mj_model.nbody}, mass={m_mj:.4f} kg')
print(f'\n[mass] diff = {abs(m_pin - m_mj):.3e} kg ({abs(m_pin - m_mj)/m_pin*100:.4f}%)')

## 2. Joint 이름·순서 매칭

Pinocchio: `universe(0)`, `floating_base(1)` 제외 실제 관절.  
MuJoCo: `free joint(0)` 제외 hinge joint.

In [ ]:
pin_joints = [pin_model.names[i] for i in range(2, pin_model.njoints)]
mj_joints  = [mj_model.joint(i).name for i in range(mj_model.njnt)
              if mj_model.jnt_type[i] != mujoco.mjtJoint.mjJNT_FREE]

print(f'Pinocchio actuated joints ({len(pin_joints)}):')
for n in pin_joints: print(f'  {n}')

print(f'\nMuJoCo actuated joints ({len(mj_joints)}):')
for n in mj_joints: print(f'  {n}')

# 두 라이브러리는 같은 joint 집합을 갖되 순서가 다를 수 있음
# (Pinocchio = URDF 이름 alphabetical, MuJoCo = body tree 순)
assert set(pin_joints) == set(mj_joints), '동일하지 않은 joint 집합!'
print('\n✅ joint 집합 일치 (순서는 다를 수 있음)')

# MuJoCo qvel/qpos를 Pinocchio 순서로 매핑하는 permutation
# perm[i] = pin 순서 i번째 joint가 mj 순서에서 갖는 인덱스
mj_to_pin_jperm = np.array([mj_joints.index(n) for n in pin_joints])
print(f'\nmj→pin joint permutation: {mj_to_pin_jperm}')

## 3. 상태 동기화 장치

MuJoCo `qpos` (free joint quat: w,x,y,z) → Pinocchio `q` (x,y,z,w) 변환.  
MuJoCo `qvel` (linear: world frame) → Pinocchio `dq` (linear: body local) 변환.

In [ ]:
def mj_to_pin_state(mj_data, R_body_to_world, jperm):
    """MuJoCo (qpos, qvel) → Pinocchio (q, dq).

    - quat:   MJ (w,x,y,z) → Pin (x,y,z,w)
    - lin_v:  MJ world → Pin body local  (R_wb.T @ v_world)
    - ang_v:  곧장 (둘 다 body local)
    - joint 순서: jperm으로 mj → pin 재배열
    """
    q  = np.zeros(pin_model.nq)
    dq = np.zeros(pin_model.nv)
    # base pos
    q[0:3] = mj_data.qpos[0:3]
    # quat
    w, x, y, z = mj_data.qpos[3:7]
    q[3:7] = [x, y, z, w]
    # joints — permute
    q[7:]  = mj_data.qpos[7:][jperm]
    # base lin vel (world → body)
    dq[0:3] = R_body_to_world.T @ mj_data.qvel[0:3]
    # base ang vel (body local — 그대로)
    dq[3:6] = mj_data.qvel[3:6]
    # joints — permute
    dq[6:]  = mj_data.qvel[6:][jperm]
    return q, dq

# 키프레임 또는 zero에서 시작 → 약간 perturb
if mj_model.nkey > 0:
    mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
else:
    mujoco.mj_resetData(mj_model, mj_data)

# yaw 30° + joint/vel perturb
rng = np.random.default_rng(0)
yaw0 = np.deg2rad(30.0)
mj_data.qpos[3] = np.cos(yaw0/2)   # MJ quat w
mj_data.qpos[6] = np.sin(yaw0/2)   # MJ quat z
mj_data.qpos[7:] += 0.05 * rng.standard_normal(mj_model.nq - 7)
mj_data.qvel[:]  = 0.1 * rng.standard_normal(mj_model.nv)
mujoco.mj_forward(mj_model, mj_data)

# base body rotation (world←body)
base_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, 'base')
R_wb = mj_data.xmat[base_id].reshape(3, 3)

q, dq = mj_to_pin_state(mj_data, R_wb, mj_to_pin_jperm)
pin.forwardKinematics(pin_model, pin_data, q, dq)
pin.computeAllTerms(pin_model, pin_data, q, dq)
pin.updateFramePlacements(pin_model, pin_data)

nv = pin_model.nv
# T = blkdiag(R_wb, I_3, P_perm) — base lin/ang은 좌표변환, joint는 permutation
T  = np.eye(nv); T[0:3, 0:3] = R_wb
# joint 블록: mj 인덱스 j → pin 인덱스 i 매핑.  T[6+i, 6+j] = 1 iff jperm[i] == j
T[6:, 6:] = 0.0
for i, j in enumerate(mj_to_pin_jperm):
    T[6+i, 6+j] = 1.0

print(f'MJ qpos[:7] (w,x,y,z) = {mj_data.qpos[:7]}')
print(f'Pin q[:7]   (x,y,z,w) = {q[:7]}')
print(f'\nR_wb (base body → world):\n{R_wb}')

## 4. 기구학 비교

- base SE3, CoM, foot frame placement, foot Jacobian (LOCAL_WORLD_ALIGNED ⇔ `mj_jac`)

In [ ]:
# 4.1 base position
p_pin = pin_data.oMf[pin_model.getFrameId('base')].translation
p_mj  = mj_data.xpos[base_id]
print(f'[base_pos]   err = {np.linalg.norm(p_pin - p_mj):.3e}')

# 4.2 CoM
com_pin = pin.centerOfMass(pin_model, pin_data, q)
com_mj  = mj_data.subtree_com[1]   # base subtree = robot 전체
print(f'[com_pos]    err = {np.linalg.norm(com_pin - com_mj):.3e}')

# 4.3 CoM Jacobian
Jcom_pin = pin.jacobianCenterOfMass(pin_model, pin_data, q)
Jcom_mj  = np.zeros((3, nv))
mujoco.mj_jacSubtreeCom(mj_model, mj_data, Jcom_mj, 1)
print(f'[J_com]      err (Jcom_mj @ T) = {np.linalg.norm(Jcom_pin - Jcom_mj @ T):.3e}')

In [ ]:
# 4.4 Foot frame placement + Jacobian
def mj_frame_jac_world(model, data, body_id, point_world):
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jac(model, data, jacp, jacr, point_world, body_id)
    return np.vstack([jacp, jacr])

for foot in ['left_foot', 'right_foot']:
    body_id = mujoco.mj_name2id(mj_model, mujoco.mjtObj.mjOBJ_BODY, foot)
    p_world = mj_data.xpos[body_id].copy()
    R_body  = mj_data.xmat[body_id].reshape(3, 3)

    # placement
    p_pin = pin_data.oMf[pin_model.getFrameId(foot)].translation
    R_pin = pin_data.oMf[pin_model.getFrameId(foot)].rotation
    err_p = np.linalg.norm(p_pin - p_world)
    err_R = np.linalg.norm(R_pin - R_body)
    print(f'[{foot:10s}] pos err = {err_p:.3e}, rot err = {err_R:.3e}')

    # Jacobian (LOCAL_WORLD_ALIGNED ⇔ mj_jac at body origin)
    J_pin = pin.getFrameJacobian(pin_model, pin_data,
                                  pin_model.getFrameId(foot),
                                  pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    J_mj  = mj_frame_jac_world(mj_model, mj_data, body_id, p_world)
    err_J = np.linalg.norm(J_pin - J_mj @ T)
    print(f'             J_world err (J_mj @ T) = {err_J:.3e}\n')

## 5. 동역학 비교

- mass matrix `M`, generalized bias `nle`, gravity `g`.
- 한 모델(같은 SolidWorks 생성)이어도 URDF·MJCF간 inertia round-off가 0.05~0.1% 나올 수 있음.

In [ ]:
# 5.1 M
M_pin = pin.crba(pin_model, pin_data, q)
# M은 upper-tri만 채움 → 대칭화
M_pin = M_pin + M_pin.T - np.diag(np.diag(M_pin))

M_mj = np.zeros((nv, nv))
mujoco.mj_fullM(mj_model, M_mj, mj_data.qM)
M_mj_to_pin = T.T @ M_mj @ T

err = np.linalg.norm(M_pin - M_mj_to_pin)
rel = err / np.linalg.norm(M_pin) * 100
print(f'[M]   ||M_pin - T^T M_mj T||_F = {err:.3e}  (relative {rel:.3f}%)')
print(f'      M_pin symmetric: {np.linalg.norm(M_pin - M_pin.T):.2e}')
print(f'      M_mj  symmetric: {np.linalg.norm(M_mj  - M_mj.T ):.2e}')

In [ ]:
# 5.2 nle = C·dq + g
nle_pin = pin.nonLinearEffects(pin_model, pin_data, q, dq)
nle_mj  = mj_data.qfrc_bias.copy()
err = np.linalg.norm(nle_pin - T.T @ nle_mj)
rel = err / np.linalg.norm(nle_pin) * 100
print(f'[nle] ||nle_pin - T^T nle_mj|| = {err:.3e}  (relative {rel:.3f}%)')

# 5.3 g — qvel=0 시 qfrc_bias
qvel_save = mj_data.qvel.copy()
mj_data.qvel[:] = 0
mujoco.mj_forward(mj_model, mj_data)
g_mj = mj_data.qfrc_bias.copy()
mj_data.qvel[:] = qvel_save
mujoco.mj_forward(mj_model, mj_data)

g_pin = pin.computeGeneralizedGravity(pin_model, pin_data, q)
err = np.linalg.norm(g_pin - T.T @ g_mj)
rel = err / np.linalg.norm(g_pin) * 100
print(f'[g]   ||g_pin - T^T g_mj||     = {err:.3e}  (relative {rel:.3f}%)')

## 6. 종합 — 예상 결과

| 양 | 예상 | 의미 |
|---|---|---|
| total mass | **불일치** (~24.9 vs ~65.3) | MJCF의 `<body name="base">`에 `<inertial>` 태그가 없어서 MuJoCo가 geom volume에서 자동 산정. URDF는 명시값 8.52 kg |
| `base_pos`, foot placement | < 1e-6 | FK 일치 |
| `com_pos` | **불일치** | base 질량이 다르므로 무게중심 위치도 다름 |
| `J_com` | **불일치** | M, m_i가 다르면 weighted Jacobian도 다름 |
| foot Jacobian | < 1e-6 | 운동학만 보면 link/joint 구조 일치 |
| `M`, `nle`, `g` | **불일치** | base 질량 차이 직접 반영 |

## ⚠️ 알려진 불일치

1. **`base` 질량**: MJCF에 `<inertial>` 누락 → MuJoCo가 collision box geom volume 기반 자동 산정 (~49 kg). URDF는 명시 8.52 kg.
   - **해결**: MJCF의 `<body name="base">` 바로 아래에 `<inertial pos="0.009896 0.004771 0.100522" mass="8.52" diaginertia="..." />` 추가 (URDF 값 복사)
   - 또는 URDF의 base mass를 MJCF 자동 산정값에 맞추는 방향 (의도가 추가 페이로드면 그럴 수 있음)

2. **Joint 순서**: Pinocchio는 URDF 이름 alphabetical (`01_right_*` 먼저), MuJoCo는 body tree 순 (`06_left_*` 먼저). 위 `T` 행렬에 permutation 포함해 보정됨.

**foot placement·Jacobian만 일치**하면 link/joint 토폴로지는 동일한 로봇 — base inertia만 손보면 dynamics도 맞출 수 있습니다.

## 7. MuJoCo viewer 스폰 — 시각 검증

론 위에 서있는 자세로 스폰되면 OK. viewer 창을 닫으면 다음 셀로 진행.

In [ ]:
if mj_model.nkey > 0:
    mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
else:
    mujoco.mj_resetData(mj_model, mj_data)

with mujoco.viewer.launch_passive(mj_model, mj_data) as viewer:
    while viewer.is_running():
        mujoco.mj_step(mj_model, mj_data)
        viewer.sync()